In [44]:
import pandas as pd
import sqlite3

## 1

In [45]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

pd.read_sql_query("SELECT * FROM test;", conn)

,uid,labname,first_commit_ts,first_view_ts
0,admin_1,None,None,2020-04-17 12:01:08.463179
1,admin_2,laba06,2020-05-19 08:22:39.690253,2020-04-28 19:09:01.362373
2,user_1,None,2020-04-26 19:06:51.939868,2020-04-26 21:53:59.624136
3,user_1,code_rvw,2020-04-26 19:06:58.949373,2020-04-26 21:53:59.624136
4,user_1,lab05s,2020-05-03 20:27:06.316718,2020-04-26 21:53:59.624136
...,...,...,...,...
83,user_30,laba04,2020-04-18 13:36:53.971502,2020-04-17 22:46:26.785035
84,user_30,laba04s,2020-04-18 14:51:37.498399,2020-04-17 22:46:26.785035
85,user_30,laba05,2020-05-01 19:31:18.375833,2020-04-17 22:46:26.785035
86,user_30,laba06,2020-05-19 17:53:15.088248,2020-04-17 22:46:26.785035


## 2 & 3

In [46]:
test_results = pd.read_sql_query("""
SELECT
    period AS time,
    AVG(delta_hours) AS avg_diff
FROM (
    SELECT
        t.uid,
        CASE
            WHEN t.first_commit_ts < t.first_view_ts THEN 'before'
            WHEN t.first_commit_ts > t.first_view_ts THEN 'after'
        END AS period,
        (d.deadlines - c.first_commit_ts) / 3600.0 AS delta_hours
    FROM datamart c
    JOIN deadlines d
        ON d.labs = c.labname
    JOIN test t
        ON t.uid = c.uid
    WHERE c.labname != 'project1'
)
WHERE period IS NOT NULL
GROUP BY period
HAVING COUNT(*) > 0;
""", conn)

test_results

,time,avg_diff
0,after,441361.664418
1,before,441383.960350


In [47]:
pd.read_sql_query("SELECT * FROM control;", conn)

,uid,labname,first_commit_ts,first_view_ts
0,None,laba04,2020-04-26 09:07:46.355781,2020-04-26 17:24:49.174370
1,None,laba04s,2020-04-26 09:08:56.887761,2020-04-26 17:24:49.174370
2,None,laba06,2020-05-19 11:15:09.236126,2020-04-26 17:24:49.174370
3,None,laba06s,2020-05-19 11:25:54.074898,2020-04-26 17:24:49.174370
4,None,project1,2020-05-11 16:11:07.874291,2020-04-26 17:24:49.174370
...,...,...,...,...
113,None,laba04s,2020-04-19 10:22:35.761944,2020-04-26 17:24:49.174370
114,None,laba05,2020-05-02 13:28:07.705193,2020-04-26 17:24:49.174370
115,None,laba06,2020-05-16 17:56:15.755553,2020-04-26 17:24:49.174370
116,None,laba06s,2020-05-16 20:01:07.900727,2020-04-26 17:24:49.174370


In [48]:
control_results = pd.read_sql_query("""
SELECT
    period AS time,
    AVG(delta_hours) AS avg_diff
FROM (
    SELECT
        c.uid,
        CASE
            WHEN c.first_commit_ts < c.first_view_ts THEN 'before'
            WHEN c.first_commit_ts > c.first_view_ts THEN 'after'
        END AS period,
        (d.deadlines - c.first_commit_ts) / 3600.0 AS delta_hours
    FROM control c
    JOIN deadlines d
        ON d.labs = c.labname
    WHERE c.labname != 'project1'
)
WHERE period IS NOT NULL
GROUP BY period
HAVING COUNT(*) > 0;
""", conn)

control_results

,time,avg_diff
0,after,441523.074975
1,before,441095.438611


## 4

In [49]:
conn.close()